# 02 · Base de conhecimento em trechos com fonte

Cada trecho (`base/trechos.jsonl`) é curto (1 a 3 frases), tem `id`, `texto`, `fonte`, `referencia`, `url`, `licenca` e `tema`. Isso permite **citar** e **auditar** cada resposta.

**Licenças:** texto de lei é domínio público; a **NBR 9050 não é reproduzida** (é protegida), usamos resumos parafraseados de cartilhas e laudos públicos de órgãos oficiais; dados do IBGE são resumos com citação; trechos do robô são de autoria da equipe.

In [1]:
import sys, json
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 200)
from rotalivre_ai import config, kb, metrics, montar, pipelines
from rotalivre_ai.runner import checar_ollama, rodar_todas
print(checar_ollama())

Ollama OK em http://localhost:11434 | LLM=qwen2.5:7b | embeddings=bge-m3


In [2]:
trechos = kb.carregar_trechos()
df = pd.DataFrame([t.__dict__ for t in trechos])
print(f"{len(df)} trechos | {df.fonte.nunique()} fontes")
tabela = (df.groupby(["fonte", "formato", "licenca"]).agg(n_trechos=("id", "count"), ids=("id", lambda s: ", ".join(s))).reset_index())
tabela

31 trechos | 14 fontes


,fonte,formato,licenca,n_trechos,ids
0,"AGU, Laudo de acessibilidade PR/Curitiba (cita itens da NBR 9050/2020)",PDF,Resumo parafraseado de publicação oficial/institucional; citar a fonte. A norma ABNT original é protegida e não é reproduzida,3,"T14, T15, T16"
1,"Agência Brasil/Band, Brasil tem 14,4 milhões de pessoas com deficiência (23/05/2025)",Notícia web,Resumo parafraseado de divulgação do IBGE (dados abertos); citar a fonte,1,T09
2,"CREA-RN, Manual do Síndico, parte 2 (calçadas e travessias)",PDF,Resumo parafraseado de publicação oficial/institucional; citar a fonte. A norma ABNT original é protegida e não é reproduzida,3,"T18, T19, T20"
3,"HU-UNIVASF/gov.br, Laudo padrão de acessibilidade (NBR 9050)",PDF,Resumo parafraseado de publicação oficial/institucional; citar a fonte. A norma ABNT original é protegida e não é reproduzida,1,T25
4,"IBGE, Censo 2022: Brasil tem 14,4 milhões de pessoas com deficiência (Agência de Notícias, 23/05/2025)",Notícia web,Resumo parafraseado de divulgação do IBGE (dados abertos); citar a fonte,4,"T05, T06, T07, T08"
5,"Instituto de Engenharia, palestra ABNT NBR 16537 / NBR 9050:2020 calçada",PDF,Resumo parafraseado de publicação oficial/institucional; citar a fonte. A norma ABNT original é protegida e não é reproduzida,1,T13
6,Lei 13.146/2015 (Lei Brasileira de Inclusão),HTML/PDF,"Domínio público (texto de lei, Lei 9.610/1998, art. 8º, IV)",3,"T01, T02, T03"
7,Lei 9.503/1997 (Código de Trânsito Brasileiro),HTML/PDF,"Domínio público (texto de lei, Lei 9.610/1998, art. 8º, IV)",1,T04
8,"Prefeitura de Arapiraca, Cartilha de calçadas (cita ABNT NBR 9050/20)",PDF,Resumo parafraseado de publicação oficial/institucional; citar a fonte. A norma ABNT original é protegida e não é reproduzida,1,T21
9,"Prefeitura de Balneário Camboriú, norma sobre calçadas (cita NBR 16537:2024)",PDF,Resumo parafraseado de publicação oficial/institucional; citar a fonte. A norma ABNT original é protegida e não é reproduzida,1,T24


In [3]:
df["palavras"] = df.texto.str.split().str.len()
print(df.palavras.describe().round(1).to_string())
print()
print(df.tema.value_counts().to_string())

count    31.0
mean     29.2
std       9.2
min      14.0
25%      23.0
50%      28.0
75%      33.0
max      57.0

tema
norma    16
robo      6
dado      5
lei       4


### Lacunas propositais
A base **não** contém (entre outros): valor de multa em R$, iluminância em lux, distância entre postes, inclinação longitudinal máxima, ciclovias, vagas de estacionamento e dados por estado. São os casos `nao_consta` que testam a recusa. Veja `base/FONTES.md` para a tabela completa de fontes.

In [4]:
texto_base = " ".join(t.texto for t in trechos).lower()
for termo in ["lux", "r$", "ciclovia", "vaga", "poste de iluminação pública", "inclinação longitudinal máxima"]:
    print(f"{termo!r:40} aparece na base? {termo in texto_base}")

'lux'                                    aparece na base? False
'r$'                                     aparece na base? False
'ciclovia'                               aparece na base? False
'vaga'                                   aparece na base? False
'poste de iluminação pública'            aparece na base? False
'inclinação longitudinal máxima'         aparece na base? False
